# channel-list-reverse-build — worked example 1: Configure generator and discriminator from one shared channel list

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `channel-list-reverse-build`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

The 'configure both halves from one list' pattern: a single `hidden_channels` list drives both networks. The **discriminator** (downsampler) iterates the list in its given order (innermost-first is chronological for it), and the **generator** (upsampler) reverses the same list with `channels[::-1]` so its blocks mirror the discriminator. One source of truth, two symmetric stacks.

## Worked solution

We want `(in_c, out_c)` pairs for each network from one `hidden_channels` list.

1. **Discriminator order.** The discriminator consumes the list as-is. We zip `hidden_channels[:-1]` with `hidden_channels[1:]` to get consecutive pairs `(in_c, out_c)`. For `[64, 128, 256, 512]` this is `[(64,128),(128,256),(256,512)]`.
2. **Generator order.** The generator must run the *same* widths in reverse, so we build `gen_channels = hidden_channels[::-1]`. The slice form (not `reversed(...)`) makes the structural symmetry visible. For the example: `[512, 256, 128, 64]`.
3. **Generator pairs.** Zip consecutive elements of `gen_channels`: `[(512,256),(256,128),(128,64)]`. Notice each generator pair is the reverse tuple of a discriminator pair, in reverse order — that is exactly the mirror symmetry we want.
4. **Why this works.** Reversing the list before pairing guarantees the generator upsamples through the same channel widths the discriminator downsampled through, so a tensor produced by the generator at any depth has a matching shape in the discriminator. Driving both from one list means you can never get them out of sync.

In [ ]:
def build_both_halves(hidden_channels):
    disc_pairs = list(zip(hidden_channels[:-1], hidden_channels[1:]))
    gen_channels = hidden_channels[::-1]
    gen_pairs = list(zip(gen_channels[:-1], gen_channels[1:]))
    return {'disc_pairs': disc_pairs, 'gen_pairs': gen_pairs}

result = build_both_halves([64, 128, 256, 512])
print('disc_pairs:', result['disc_pairs'])
print('gen_pairs :', result['gen_pairs'])